In [2]:
!pip install -q zarr numcodecs


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 6.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 89.8 MB/s eta 0:00:00


In [1]:
# ==============================
# ERA5 PREPROCESSING (SAFE VERSION)
# ==============================

from google.colab import drive
drive.mount('/content/drive')

import xarray as xr
import numpy as np
import os

# ------------------------------
# CONFIG
# ------------------------------
BASE_PATH = "/content/drive/MyDrive/Indian Weather Prediction/datasets"
OUT_PATH  = "/content/drive/MyDrive/Indian Weather Prediction/datasets/processed.zarr"

TRAIN_YEARS = list(range(2015, 2024))
VARIABLES = ["t2m", "d2m", "sp", "msl", "u10", "v10", "tp"]
NORMALIZE_VARS = ["t2m", "d2m", "sp", "msl", "u10", "v10"]

# ------------------------------
# DATA LOADER (FROM NOTEBOOK 1)
# ------------------------------
def load_year(year):
    year_path = os.path.join(BASE_PATH, str(year))
    ds_inst = xr.open_dataset(
        os.path.join(year_path, "data_stream-oper_stepType-instant.nc"),
        chunks={}
    )
    ds_acc = xr.open_dataset(
        os.path.join(year_path, "data_stream-oper_stepType-accum.nc"),
        chunks={}
    )
    ds = xr.merge([ds_inst, ds_acc], compat="no_conflicts")
    return ds[VARIABLES]

# ------------------------------
# LOAD & CONCAT YEARS (LAZY)
# ------------------------------
datasets = []
for y in TRAIN_YEARS:
    print(f"Loading {y}")
    datasets.append(load_year(y))

ds = xr.concat(datasets, dim="valid_time")

# ------------------------------
# CHUNK IMMEDIATELY (CRITICAL)
# ------------------------------
ds = ds.chunk({
    "valid_time": 200,
    "latitude": 50,
    "longitude": 50
})

print("Dataset loaded & chunked safely")

# ------------------------------
# TARGET ENGINEERING (RAIN)
# ------------------------------
ds["tp"] = np.log1p(ds["tp"])

print("Rainfall log-transform applied")

# ------------------------------
# COMPUTE NORMALIZATION STATS
# (ON SUBSET — MEMORY SAFE)
# ------------------------------
print("Computing normalization statistics...")

# Use first ~2 years only (≈ 3000 timesteps)
ds_stats = ds.isel(valid_time=slice(0, 3000))

norm_stats = {}

for var in NORMALIZE_VARS:
    mean = ds_stats[var].mean(
        dim=["valid_time", "latitude", "longitude"]
    ).compute()

    std = ds_stats[var].std(
        dim=["valid_time", "latitude", "longitude"]
    ).compute()

    norm_stats[var] = {
        "mean": float(mean),
        "std": float(std)
    }
    print(f"{var}: mean={mean.values:.4f}, std={std.values:.4f}")

print("Normalization stats computed safely")

# ------------------------------
# APPLY NORMALIZATION (LAZY)
# ------------------------------
for var in NORMALIZE_VARS:
    ds[var] = (ds[var] - norm_stats[var]["mean"]) / norm_stats[var]["std"]

print("Normalization applied (lazy)")

# ------------------------------
# FINAL CHUNKING (TRAINING OPTIMAL)
# ------------------------------
ds = ds.chunk({
    "valid_time": 100,
    "latitude": -1,
    "longitude": -1
})

# ------------------------------
# SAVE AS ZARR (STREAMABLE)
# ------------------------------
print("Saving processed dataset to Zarr... (this may take ~10 minutes)")

ds.to_zarr(OUT_PATH, mode="w")

print("✅ PREPROCESSING COMPLETE")
print(f"Saved to: {OUT_PATH}")

# ------------------------------
# SANITY CHECK
# ------------------------------
ds_check = xr.open_zarr(OUT_PATH)
print(ds_check)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading 2015
Loading 2016
Loading 2017
Loading 2018
Loading 2019
Loading 2020
Loading 2021
Loading 2022
Loading 2023
Dataset loaded & chunked safely
Rainfall log-transform applied
Computing normalization statistics...
t2m: mean=292.9800, std=14.1032
d2m: mean=286.2926, std=15.4470
sp: mean=89760.5781, std=17107.9297
msl: mean=101158.3516, std=646.3034
u10: mean=0.8466, std=3.1936
v10: mean=0.1836, std=2.7646
Normalization stats computed safely
Normalization applied (lazy)
Saving processed dataset to Zarr... (this may take ~10 minutes)


/usr/local/lib/python3.12/dist-packages/zarr/core/dtype/npy/string.py:249: UnstableSpecificationWarning: The data type (FixedLengthUTF32(length=4, endianness='little')) does not have a Zarr V3 specification. That means that the representation of arrays saved with this data type may change without warning in a future version of Zarr Python. Arrays stored with this data type may be unreadable by other Zarr libraries. Use this data type at your own risk! Check https://github.com/zarr-developers/zarr-extensions/tree/main/data-types for the status of data type specifications for Zarr V3.
  v3_unstable_dtype_warning(self)
/usr/local/lib/python3.12/dist-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


✅ PREPROCESSING COMPLETE
Saved to: /content/drive/MyDrive/Indian Weather Prediction/datasets/processed.zarr
<xarray.Dataset> Size: 6GB
Dimensions:     (valid_time: 13148, latitude: 125, longitude: 121)
Coordinates:
  * valid_time  (valid_time) datetime64[ns] 105kB 2015-01-01 ... 2023-12-31T1...
  * latitude    (latitude) float64 1kB 37.0 36.75 36.5 36.25 ... 6.5 6.25 6.0
  * longitude   (longitude) float64 968B 68.0 68.25 68.5 ... 97.5 97.75 98.0
    expver      (valid_time) <U4 210kB dask.array<chunksize=(100,), meta=np.ndarray>
    number      int64 8B ...
Data variables:
    d2m         (valid_time, latitude, longitude) float32 795MB dask.array<chunksize=(100, 125, 121), meta=np.ndarray>
    msl         (valid_time, latitude, longitude) float32 795MB dask.array<chunksize=(100, 125, 121), meta=np.ndarray>
    sp          (valid_time, latitude, longitude) float32 795MB dask.array<chunksize=(100, 125, 121), meta=np.ndarray>
    t2m         (valid_time, latitude, longitude) float32 795M